# Deep Network Development, 2026 Autumn - Week 7 Practice - **Exercise sheet**

## Exercises

---

### Exercise 7.1: Predicting body mass and species

#### Problem Description

Continue with the [Palmer Penguins dataset](https://allisonhorst.github.io/palmerpenguins/).
Train a regression network to predict body mass and a three-class network to predict species.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

DATA_URL = (
    'https://raw.githubusercontent.com/allisonhorst/palmerpenguins/'
    '156daa4301838d9fdcc5b018b29b9149bf975552/inst/extdata/penguins.csv'
)
species_dict = {'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2}
sex_dict = {'female': 0, 'male': 1}
SPECIES_NAMES = np.array(['Adelie', 'Chinstrap', 'Gentoo'])
COLUMN_NAMES = np.array(['species', 'bill_length_mm', 'bill_depth_mm',
                         'flipper_length_mm', 'body_mass_g', 'sex'])

CATEGORY_CONVERTERS = {'species': species_dict.get, 'sex': sex_dict.get}
FEATURE_COLUMNS = [1, 2, 3, 5]
FEATURE_NAMES = COLUMN_NAMES[FEATURE_COLUMNS]


#### Task 1: Loading with Pandas and converting to NumPy

Load `DATA_URL` into the DataFrame `dataset_df` using `pd.read_csv`.
Use `COLUMN_NAMES` for `usecols` and `CATEGORY_CONVERTERS` for `converters`.

Convert `dataset_df` to the NumPy array `dataset_arr` with `np.float32` dtype using `.to_numpy()`.


In [ ]:
# Implement your solution below







# Prints
print(dataset_df.head())
print('Shape:', dataset_df.shape)
print('Column dtypes:')
print(dataset_df.dtypes)
print('NumPy:', dataset_arr.shape, dataset_arr.dtype)

# Test cases
assert dataset_df.shape == (344, 6)
assert np.array_equal(dataset_df.columns.to_numpy(), COLUMN_NAMES)
assert dataset_df.isna().sum().sum() == 19
assert np.allclose(dataset_df.iloc[0].to_numpy(dtype=np.float32), [0, 39.1, 18.7, 181, 3750, 1])
assert dataset_arr.shape == (344, 6) and dataset_arr.dtype == np.float32
assert np.isnan(dataset_arr).sum() == 19
print('Task 1 passed!')


#### Provided setup

Apply the supplied fixed ordering to `dataset_arr`.


In [ ]:
# Provided fixed mixed ordering
row_order = np.random.default_rng(7).permutation(len(dataset_arr))
dataset_arr = dataset_arr[row_order]

print('Dataset:', dataset_arr.shape)
print('Input columns:', FEATURE_NAMES)
assert dataset_arr.shape == (344, 6)
assert np.isnan(dataset_arr).sum() == 19


#### Task 2: Inputs, targets, and PyTorch tensors

Create `complete_row_mask`, selecting rows of `dataset_arr` with no NaNs.
Create `clean_dataset_arr` from these rows. Keep `dataset_arr` unchanged.

Create `inputs_arr` from `FEATURE_COLUMNS` of `clean_dataset_arr`, then create:

- `X_class` and `X_reg`: inputs, `torch.float32`, shape `(333, 4)`.
- `y_class`: species indices (column 0), `torch.long`, shape `(333,)`.
- `y_reg`: body masses in grams (column 4), `torch.float32`, shape `(333, 1)`.


In [ ]:
# Implement your solution below







# Prints
print('Rows kept / removed:', len(clean_dataset_arr), len(dataset_arr) - len(clean_dataset_arr))
print('Classification:', X_class.shape, X_class.dtype, y_class.shape, y_class.dtype)
print('Regression:', X_reg.shape, X_reg.dtype, y_reg.shape, y_reg.dtype)

# Test cases
assert complete_row_mask.shape == (344,) and complete_row_mask.dtype == np.bool_
assert np.array_equal(complete_row_mask, ~np.isnan(dataset_arr).any(axis=1))
assert clean_dataset_arr.shape == (333, 6) and np.isfinite(clean_dataset_arr).all()
assert np.array_equal(clean_dataset_arr, dataset_arr[complete_row_mask])
assert inputs_arr.shape == (333, 4) and np.isfinite(inputs_arr).all()
assert np.isnan(dataset_arr).sum() == 19
assert X_class.shape == X_reg.shape == (333, 4)
assert y_class.shape == (333,) and y_reg.shape == (333, 1)
assert X_class.dtype == X_reg.dtype == y_reg.dtype == torch.float32
assert y_class.dtype == torch.long
assert torch.equal(X_reg, X_class)
assert torch.equal(y_class, torch.tensor(clean_dataset_arr[:, 0], dtype=torch.long))
assert np.allclose(y_reg.numpy().ravel(), clean_dataset_arr[:, 4])
print('Task 2 passed!')


#### Task 3: Train, validation, and test splits

Split each task's inputs and targets by slicing: **70% training, 15% validation, and the remaining 15% testing**.
For each dataset, calculate the training and validation counts with `int(fraction * number_of_rows)`;
assign all remaining rows to testing.

Create `X_reg_train`, `X_reg_val`, `X_reg_test` and the corresponding `y_reg_*` tensors.
Use the same naming pattern for classification: `X_class_*`, `y_class_*`.


In [ ]:
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15

# Implement your solution below







# Prints
print('Regression split sizes:', len(X_reg_train), len(X_reg_val), len(X_reg_test))
print('Classification split sizes:', len(X_class_train), len(X_class_val), len(X_class_test))
print('Classification label counts (train / val / test):')
for labels in [y_class_train, y_class_val, y_class_test]:
    print(torch.bincount(labels, minlength=3))

# Test cases
for X, y, X_splits, y_splits in [
    (X_reg, y_reg, [X_reg_train, X_reg_val, X_reg_test], [y_reg_train, y_reg_val, y_reg_test]),
    (X_class, y_class, [X_class_train, X_class_val, X_class_test], [y_class_train, y_class_val, y_class_test])
]:
    n_train = int(TRAIN_FRACTION * len(X))
    n_val = int(VAL_FRACTION * len(X))
    assert [len(part) for part in X_splits] == [n_train, n_val, len(X) - n_train - n_val]
    assert [len(part) for part in y_splits] == [len(part) for part in X_splits]
    assert torch.equal(torch.cat(X_splits), X)
    assert torch.equal(torch.cat(y_splits), y)
assert all(torch.all(torch.bincount(y, minlength=3) > 0)
           for y in [y_class_train, y_class_val, y_class_test])
print('Task 3 passed!')


#### Task 4: A regression network

Define `RegressionNetwork`, a subclass of `nn.Module`, with two linear layers:
`self.hidden` (4 inputs, 16 outputs) and `self.output` (16 inputs, 1 output).
In `forward(x)`, apply the hidden layer, ReLU, and then the output layer with no output activation.

Create `reg_model`, `reg_loss_fn` using `nn.MSELoss`, and `reg_optimizer` using Adam
with learning rate `0.01` and `reg_model.parameters()`.


In [ ]:
torch.manual_seed(7)

# Implement your solution below







# Prints
print('Prediction shape:', reg_model(X_reg_train[:5]).shape)
print('Initial training MSE:', reg_loss_fn(reg_model(X_reg_train), y_reg_train).item())

# Test cases
assert isinstance(reg_model, nn.Module)
assert reg_model.hidden.in_features == 4 and reg_model.hidden.out_features == 16
assert reg_model.output.in_features == 16 and reg_model.output.out_features == 1
assert reg_model(X_reg_train[:5]).shape == (5, 1)
assert isinstance(reg_loss_fn, nn.MSELoss)
assert {id(p) for g in reg_optimizer.param_groups for p in g['params']} == {id(p) for p in reg_model.parameters()}
assert torch.isclose(reg_loss_fn(torch.tensor([[1.], [3.]]),
                               torch.tensor([[2.], [5.]])), torch.tensor(2.5))
print('Task 4 passed!')


#### Task 5: Manual mini-batch regression training

Train for `REG_EPOCHS` epochs with manual batches of size `BATCH_SIZE`.

At each epoch, call `reg_model.train()`. For each training batch, reset gradients,
predict, calculate the loss, collect its detached value, call `backward()`, and update parameters.
Use `.mean()` on the collected batch losses and append the result to `reg_train_losses`.

Call `reg_model.eval()` and use `torch.no_grad()` to collect validation batch losses in the same way.
Append their mean to `reg_val_losses`. Print the training and validation losses only after every 10th epoch (10, 20, 30, ...).


In [ ]:
BATCH_SIZE = 32
REG_EPOCHS = 200
reg_train_losses, reg_val_losses = [], []
reg_initial_weights = reg_model.hidden.weight.detach().clone()

# Implement your solution below







# Prints
plt.plot(reg_train_losses, label='Training')
plt.plot(reg_val_losses, label='Validation')
plt.xlabel('Epochs')
plt.ylabel("MSE loss")
plt.ylim(bottom=0)
plt.legend()
plt.show()

# Test cases
assert len(reg_train_losses) == len(reg_val_losses) == REG_EPOCHS
assert np.isfinite(reg_train_losses).all() and np.isfinite(reg_val_losses).all()
assert min(reg_train_losses) >= 0 and min(reg_val_losses) >= 0
assert not torch.equal(reg_initial_weights, reg_model.hidden.weight.detach())
print('Task 5 passed!')


#### Task 6: Regression metrics and example predictions

Implement `mean_absolute_error(predictions, targets)` returning a scalar tensor:

$$\mathrm{MAE} = \frac{1}{N} \sum_{i=1}^{N} |\hat{y}_i - y_i|$$

Here, $N$ is the number of examples, $\hat{y}_i$ is a prediction, and $y_i$ is its target.
Calculate `reg_val_predictions` without gradients, then `reg_val_mae`.

Create `reg_example_predictions` from the first five validation predictions, as a 1D tensor in grams.


In [ ]:
# Implement your solution below







# Prints
print('Validation MAE (g):', reg_val_mae.item())
print('True / predicted validation mass (g):')
for true, predicted in zip(y_reg_val[:5, 0], reg_example_predictions):
    print(f'{true.item():.0f} / {predicted.item():.1f}')

# Test cases
assert torch.isclose(mean_absolute_error(torch.tensor([[1.], [4.]]),
                                        torch.tensor([[3.], [3.]])), torch.tensor(1.5))
assert reg_val_predictions.shape == y_reg_val.shape and not reg_val_predictions.requires_grad
assert reg_example_predictions.shape == (5,)
assert reg_val_mae.ndim == 0
assert torch.isfinite(reg_val_mae)
assert torch.allclose(reg_val_mae, (reg_val_predictions - y_reg_val).abs().mean())
print('Task 6 passed!')


#### Task 7: A multi-class network

Define `ClassificationNetwork`, a subclass of `nn.Module`, with `self.hidden` (4 inputs, 16 outputs)
and `self.output` (16 inputs, 3 outputs). In `forward(x)`, apply ReLU after the hidden layer
and no output activation. Return one raw score (logit) per species.

Create `class_model`, `class_loss_fn` using `nn.CrossEntropyLoss`, and `class_optimizer`
using Adam with learning rate `0.001` and `class_model.parameters()`.
Use integer class indices directly as targets.


In [ ]:
torch.manual_seed(7)

# Implement your solution below







# Prints
print('Logit shape:', class_model(X_class_train[:5]).shape)
print('Target shape and dtype:', y_class_train[:5].shape, y_class_train.dtype)
print('Initial cross-entropy:', class_loss_fn(class_model(X_class_train), y_class_train).item())

# Test cases
assert isinstance(class_model, nn.Module)
assert class_model.hidden.in_features == 4 and class_model.hidden.out_features == 16
assert class_model.output.in_features == 16 and class_model.output.out_features == 3
assert class_model(X_class_train[:5]).shape == (5, 3)
assert isinstance(class_loss_fn, nn.CrossEntropyLoss)
assert {id(p) for g in class_optimizer.param_groups for p in g['params']} == {id(p) for p in class_model.parameters()}
assert torch.isclose(class_loss_fn(torch.zeros(2, 3), torch.tensor([0, 2])),
                     torch.tensor(np.log(3), dtype=torch.float32))
print('Task 7 passed!')


#### Task 8: Manual mini-batch classification training

Train for `CLASS_EPOCHS` epochs with manual batches of size `BATCH_SIZE`.

At each epoch, call `class_model.train()`. For each training batch, reset gradients,
predict, calculate the loss, collect its detached value, call `backward()`, and update parameters.
Use `.mean()` on the collected batch losses and append the result to `class_train_losses`.

Call `class_model.eval()` and use `torch.no_grad()` to collect validation batch losses in the same way.
Append their mean to `class_val_losses`. Print the training and validation losses only after every 10th epoch (10, 20, 30, ...).


In [ ]:
# Reuse BATCH_SIZE from Task 5
CLASS_EPOCHS = 200
class_train_losses, class_val_losses = [], []
class_initial_weights = class_model.hidden.weight.detach().clone()

# Implement your solution below







# Prints
plt.plot(class_train_losses, label='Training')
plt.plot(class_val_losses, label='Validation')
plt.xlabel('Epochs')
plt.ylabel("Cross-entropy loss")
plt.ylim(bottom=0)
plt.legend()
plt.show()

# Test cases
assert len(class_train_losses) == len(class_val_losses) == CLASS_EPOCHS
assert np.isfinite(class_train_losses).all() and np.isfinite(class_val_losses).all()
assert min(class_train_losses) >= 0 and min(class_val_losses) >= 0
assert not torch.equal(class_initial_weights, class_model.hidden.weight.detach())
print('Task 8 passed!')


#### Task 9: Classification accuracy and predictions

Implement `classification_accuracy(logits, targets)`: select the largest score for each row
and return the proportion of correct predictions as a scalar tensor.

Calculate `class_test_logits` on the classification test split without gradients, then
`class_test_accuracy`. Create `class_example_predictions`: the predicted integer species
labels for the first five test examples.
Report accuracy on a 0-1 scale.


In [ ]:
# Implement your solution below







# Prints
print('Test classification accuracy:', class_test_accuracy.item())
print('True / predicted test species:')
for true, predicted in zip(y_class_test[:5], class_example_predictions):
    print(SPECIES_NAMES[true.item()], '/', SPECIES_NAMES[predicted.item()])

# Test cases
toy_logits = torch.tensor([[4., 1., 0.], [0., 3., 1.], [2., 0., 4.], [1., 5., 2.]])
toy_targets = torch.tensor([0, 2, 2, 1])
assert torch.isclose(classification_accuracy(toy_logits, toy_targets), torch.tensor(0.75))
assert class_test_logits.shape == (len(X_class_test), 3) and not class_test_logits.requires_grad
assert class_example_predictions.shape == (5,) and class_example_predictions.dtype == torch.long
assert torch.equal(class_example_predictions, class_test_logits[:5].argmax(dim=1))
assert class_test_accuracy.ndim == 0 and 0 <= class_test_accuracy.item() <= 1
assert torch.allclose(class_test_accuracy,
                      (class_test_logits.argmax(dim=1) == y_class_test).float().mean())
print('Task 9 passed!')


#### Quick recap: outputs, targets, and losses

For a batch of size `B`:

| Task | Output shape | Target shape / dtype | Loss | Prediction |
|---|---|---|---|---|
| Regression | `(B, 1)` | `(B, 1)` / `torch.float32` | `nn.MSELoss` | Scalar output directly |
| Binary classification | `(B, 1)`, after Sigmoid | `(B, 1)` / `torch.float32` | `nn.BCELoss` | Probability >= 0.5 predicts 1 |
| Multi-class classification (3 classes) | `(B, 3)` | `(B,)` / `torch.long` | `nn.CrossEntropyLoss` | `argmax(dim=1)` |


**Useful documentation links:**

- [MSELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.MSELoss.html)
- [CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)
- [BCELoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)
